# TinyGPT from scratch — interactive demo

A small GPT written from scratch in pure PyTorch and trained for **7 minutes on 4 CPU threads** per run,
to ask: **does the tokenizer matter?** Runs: `char`, `bpe512`, `bpe2048`, and `bpe512_nopos`
(no positional embeddings). Runs are compared by validation **bits per character (BPC)** on identical text.

Run the cells top to bottom (Runtime → Run all). No GPU needed.

In [ ]:
# ==== SET THIS ===========================================================
GITHUB_USERNAME = "Devyansh-Choudhary"   # <-- your GitHub username (owner of the fork)
# =========================================================================
REPO = "tinygpt-from-scratch"

import os, subprocess
if os.path.isdir("tinygpt") and os.path.isfile("SPEC.md"):
    print("Already inside the repo:", os.getcwd())
else:
    if not os.path.isdir(REPO):
        url = f"https://github.com/{GITHUB_USERNAME}/{REPO}"
        print("Cloning", url)
        subprocess.run(["git", "clone", "--depth", "1", url, REPO], check=True)
    os.chdir(REPO)
    print("cwd:", os.getcwd())

In [ ]:
# Colab already ships torch/numpy/matplotlib/ipywidgets; this only fills gaps.
%pip install -q -r requirements.txt

## Load the trained checkpoints

In [ ]:
import glob, os
import torch
from IPython.display import Image, display

from tinygpt.evaluate import load_checkpoint

torch.manual_seed(1337)
torch.set_num_threads(4)

RUN_ORDER = ["char", "bpe512", "bpe2048", "bpe512_nopos"]
models = {}   # run_name -> (model, tokenizer, ckpt_dict)
for path in sorted(glob.glob("checkpoints/*.pt")):
    run = os.path.splitext(os.path.basename(path))[0]
    try:
        models[run] = load_checkpoint(path)
    except Exception as e:
        print(f"could not load {path}: {type(e).__name__}: {e}")
RUNS = sorted(models, key=lambda r: (RUN_ORDER.index(r) if r in RUN_ORDER else 99, r))

if not RUNS:
    print("No checkpoints/*.pt found. Train one first, e.g. (2-minute budget):\n"
          "  !python -m tinygpt.train --run_name char --tokenizer char --time_budget_min 2")
else:
    print(f"{'run':<14}{'tokenizer':<11}{'vocab':>7}{'params (non-emb)':>18}{'final val BPC':>15}")
    for run in RUNS:
        model, tok, ck = models[run]
        bpc = ck.get("final_val_bpc", float("nan"))
        print(f"{run:<14}{tok.name:<11}{tok.vocab_size:>7}{model.num_params():>18,}{bpc:>15.3f}")

## Training curves

Validation BPC against training wall-clock time (lower is better), and the final BPC of every run.

In [ ]:
from viz.curves import plot_bpc_curves, plot_final_bpc

os.makedirs("results/figures", exist_ok=True)
try:
    display(Image(plot_bpc_curves("results", "results/figures/bpc_curves.png")))
    display(Image(plot_final_bpc("results", "results/figures/final_bpc.png", checkpoint_dir="checkpoints")))
except FileNotFoundError as e:
    print("No training logs yet:", e)

## Samples written during training (prompt: "Once upon a time")

In [ ]:
for path in sorted(glob.glob("results/*/samples.txt")):
    run = path.split(os.sep)[-2]
    text = open(path).read()
    last = text.rsplit("=== step", 1)   # show only the final sample of each run
    print(f"##### {run}  (final sample)\n" + ("=== step" + last[1] if len(last) == 2 else text)[:1200])
    print()

## Generate your own text

Pick a run, type a prompt, set **temperature** (lower = safer, higher = wilder) and **top_k**
(0 = no top-k filtering), then press **Generate**.

In [ ]:
import ipywidgets as widgets

try:  # Colab needs this for third-party widget rendering; harmless elsewhere
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

run_dd = widgets.Dropdown(options=RUNS, value=RUNS[0] if RUNS else None, description="run")
prompt_tb = widgets.Textarea(value="Once upon a time", description="prompt",
                             layout=widgets.Layout(width="90%", height="70px"))
temp_sl = widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.05, description="temperature")
topk_sl = widgets.IntSlider(value=40, min=0, max=200, step=1, description="top_k")
len_sl = widgets.IntSlider(value=200, min=10, max=600, step=10, description="new tokens")
go_btn = widgets.Button(description="Generate", button_style="primary", icon="play")
out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="6px"))

def on_generate(_):
    with out:
        out.clear_output()
        if not run_dd.value:
            print("No checkpoints loaded.")
            return
        model, tok, _ = models[run_dd.value]
        ids = tok.encode(prompt_tb.value) or tok.encode("\n")
        idx = torch.tensor([ids], dtype=torch.long)
        y = model.generate(idx, max_new_tokens=len_sl.value, temperature=temp_sl.value,
                           top_k=topk_sl.value or None)
        print(tok.decode(y[0].tolist()))

go_btn.on_click(on_generate)
display(widgets.VBox([run_dd, prompt_tb, temp_sl, topk_sl, len_sl, go_btn, out]))

## Attention heatmaps for a custom sentence

Rows of each map are query tokens; columns are the tokens they attend to (grey = future, masked).
Tick labels are the decoded tokens: `␣` is a space and `↵` a newline. The second figure zooms in on the
single most distinctive head, with a one-line guess at what it does.

In [ ]:
from viz.attention import plot_attention, plot_interesting_head

SENTENCE = "Once upon a time, there was a little girl named Lily. She loved to play in the park."
RUN = RUNS[0] if RUNS else None   # e.g. "bpe512"

if RUN:
    model, tok, _ = models[RUN]
    display(Image(plot_attention(model, tok, SENTENCE, f"results/figures/attention_{RUN}.png",
                                 title=f"Attention by layer and head  ·  run {RUN}")))
    path, caption = plot_interesting_head(model, tok, SENTENCE, f"results/figures/attention_{RUN}_head.png")
    display(Image(path))
    print(caption)

## Roofline (where each op sits against this machine's compute / bandwidth limits)

In [ ]:
if os.path.isfile("roofline/roofline.png"):
    display(Image("roofline/roofline.png"))
else:
    print("roofline/roofline.png not found; run: python -m roofline.run")

---
Regenerate every figure from the command line with `python -m viz.make_all` (writes to `results/figures/`).